# 36. Stress Severity Ladder와 Glare Counterfactual

2-2장의 severe stress는 Dice 0 바닥 효과가 있으므로 mild/moderate/severe ladder를 새로 만들어 원인 분리가 가능한 평가셋을 구성합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch3_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/3장/ch3_utils.py")) + list(Path.cwd().glob("**/ch3_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "3장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch3_utils import *

paths = find_ch3_paths()
set_korean_font()
set_seed(31)
paths

Chapter3Paths(chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), chapter2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data'), stress_ladder_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data/synthetic_metal_stress_ladder'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/manifests'), ch2_2_runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'))

## 36-1. stress ladder 데이터셋 생성

In [2]:
STRESS_PER_CELL = 2
stress_root = generate_stress_ladder_dataset(
    image_size=128,
    per_cell=STRESS_PER_CELL,
    seed=3307,
    overwrite=False,
    include_no_glare_pairs=True,
)
stress_samples = load_stress_ladder_samples(stress_root)
stress_manifest = create_stress_ladder_manifest(stress_samples)
print(stress_root)
print(stress_manifest, "rows=", len(pd.read_csv(stress_manifest)))
display(
    stress_samples.groupby(["stress_severity", "counterfactual_type", "color_group"])
    .size()
    .reset_index(name="count")
    .head(20)
)

C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\data\synthetic_metal_stress_ladder
C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\manifests\stress_ladder\eval_stress_ladder_manifest.csv rows= 480


,stress_severity,counterfactual_type,color_group,count
0,mild,no_glare,blue,16
1,mild,no_glare,green,16
2,mild,no_glare,neutral,16
3,mild,no_glare,purple,16
4,mild,no_glare,red,16
5,mild,original_stress,blue,16
6,mild,original_stress,green,16
7,mild,original_stress,neutral,16
8,mild,original_stress,purple,16
9,mild,original_stress,red,16


## 36-2. baseline 모델 stress ladder 평가

In [3]:
RUN_STRESS_EVAL = True
registry = discover_ch3_model_registry()
selected = registry[
    (registry["family"] == "baseline")
    & (registry["variant"] == "baseline_no_aug")
    & (registry["model_seed"] == 0)
    & (registry["checkpoint_exists"])
]
if selected.empty:
    raise FileNotFoundError("baseline seed_0 checkpoint가 없습니다.")

out_dir = paths.runs_root / "stress_ladder"
out_dir.mkdir(parents=True, exist_ok=True)
if RUN_STRESS_EVAL:
    eval_dir = out_dir / "baseline_no_aug_seed_0"
    evaluate_saved_model_on_manifest(selected.iloc[0]["run_dir"], stress_manifest, eval_dir, batch_size=8)
    metrics = pd.read_csv(eval_dir / "sample_metrics.csv")
    stress_meta = pd.read_csv(stress_manifest)[["sample_id", "stress_severity", "counterfactual_type"]]
    metrics = metrics.merge(stress_meta, on="sample_id", how="left")
    metrics.to_csv(out_dir / "stress_ladder_metrics.csv", index=False, encoding="utf-8-sig")
    display(
        metrics.groupby(["stress_severity", "counterfactual_type"])[["target_dice", "target_fnr"]]
        .mean()
        .reset_index()
    )
else:
    print("RUN_STRESS_EVAL=False: 데이터셋과 manifest만 생성했습니다. 평가하려면 True로 바꾸세요.")

C:\Users\준승\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,stress_severity,counterfactual_type,target_dice,target_fnr
0,mild,no_glare,0.455554,0.573971
1,mild,original_stress,0.430865,0.577169
2,moderate,no_glare,0.010517,0.993153
3,moderate,original_stress,0.017503,0.986267
4,severe,no_glare,0.012338,0.988193
5,severe,original_stress,0.009299,0.993449


## 36-3. glare/no-glare 비교 해석

In [4]:
metrics_path = paths.runs_root / "stress_ladder" / "stress_ladder_metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(
        metrics.groupby(["stress_severity", "counterfactual_type", "color_group"])[
            ["target_dice", "target_fnr"]
        ]
        .mean()
        .reset_index()
    )
else:
    print("stress_ladder_metrics.csv가 없습니다. 위 셀에서 RUN_STRESS_EVAL=True로 평가를 실행하세요.")

,stress_severity,counterfactual_type,color_group,target_dice,target_fnr
0,mild,no_glare,blue,0.544155,0.478080
1,mild,no_glare,green,0.638125,0.404640
2,mild,no_glare,neutral,0.561881,0.498247
3,mild,no_glare,purple,0.533611,0.488890
4,mild,no_glare,red,0.000000,1.000000
5,mild,original_stress,blue,0.692865,0.317656
6,mild,original_stress,green,0.559069,0.455951
7,mild,original_stress,neutral,0.636477,0.360956
8,mild,original_stress,purple,0.265914,0.751283
9,mild,original_stress,red,0.000000,1.000000
